In [1]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [3]:
transform_data = transforms.Compose([transforms.ToTensor(), transforms.Normalize(0.1307, 0.3081)])

In [4]:
train_dataset= datasets.MNIST('./datasets', train=True, transform=transform_data, download=True)

100.0%
100.0%
100.0%
100.0%


In [5]:
test_dataset= datasets.MNIST('./datasets', train=False, transform=transform_data, download=True)

In [ ]:
train_loader = DataLoader(train_dataset,32, shuffle=True)
test_loader = DataLoader(test_loader, 32, shuffle=False)

The Devlopment of the Fatigue Calculation Process Functions

In [2]:
import torch 
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
"""
This file contains the main implementation of the fatigue index or the CSI 
that will be later used in step function of the Adam Optimizer, when we override the step function  

"""



"""
This function makes sure, to exclude from calculations any parmeters that have no gradients
in both the current and previous gradients vectors. This is to consider the cases where there is
any freezing parameters that are with no gradients.
"""
def frozen_parameters_calculate(prev_grad: np.ndarray,  current_grad:np.ndarray):
    frozen_parameters_index =  []
    for i in range (len(prev_grad)):
        if (prev_grad[i]==0 and current_grad[i]==0):
            frozen_parameters_index.append(i)
    return frozen_parameters_index

def tensor_to_array(tensor: torch.Tensor):
    arr = tensor.clone().detach().cpu().numpy()
    return arr





def gradient_conflict(prev_grad: torch.Tensor,  current_grad:torch.Tensor):


    prev_grad= tensor_to_array(prev_grad)
    current_grad= tensor_to_array(current_grad)
    frozen_indices = frozen_parameters_calculate(prev_grad=prev_grad, current_grad=current_grad)

    keep_indices = [i for i in range(len(prev_grad)) if i not in frozen_indices]

    
    
    filtered_prev_grad = prev_grad[keep_indices]

    filtered_current_grad = current_grad[keep_indices]

    filtered_prev_grad=filtered_prev_grad.reshape(1,-1)

    filtered_current_grad=filtered_current_grad.reshape(1,-1)


    similarity= cosine_similarity(filtered_current_grad, filtered_prev_grad)
    distance = 1-similarity

    return distance[0][0]





def error_oscillation_calculate():
    pass

In [12]:
print(gradient_conflict(torch.Tensor([1,2]), torch.Tensor([1,2])))

5.9604645e-08


Implementing the second signal, the Error direction value osciallation

In [14]:
class LossTracker():
    def __init__(self, window_size=20):
        self.history = []
    def log(self, loss_value):
        self.history.append(loss_value)
    def get_history(self):
        return self.history

In [15]:

def error_oscillation_calculate(loss_array_tracker: LossTracker):
    lossArray = loss_array_tracker.get_history()
    if len(lossArray)<1:
        return "NOT ENOUGH VALUES"
    deltas = []
    for i in range(len(lossArray)-1):
        deltas.append(lossArray[i+1]-lossArray[i])

    numberOfFlipsOfSign=0    
    for i in range(len(deltas)-1):
        if deltas[i]*deltas[i+1]<0:
            numberOfFlipsOfSign+=1
    normalized = numberOfFlipsOfSign/(len(deltas)-1)
    return normalized
